> **中文备注(自用)**
>
> **本文件 = 检索部分(Stage 1–3),与 `final_retrieve_classify` 的检索代码一致。**
>
> 流程:双路 TF-IDF 召回 → CrossEncoder 精排 → 分数融合 + gap 自适应选择,最后**导出** `retrieved_train.json` / `retrieved_dev.json`(claim + 选出的证据 + 标签)。
>
> 目的:检索只跑一次,导出后 LLM 分类器 notebook 直接加载,**不必重跑检索**,且保证 LLM 与 DeBERTa 用完全相同的 retrieved evidence(对照公平)。

# 1. Dataset Processing & Dual TF-IDF Retrieval

In [ ]:
# ============================================================
# 1. DataSet Processing
# ============================================================
# Install the required NLP and ML libraries in the notebook environment.

!pip install -q sentence-transformers scikit-learn tqdm gdown transformers

import os
import json
import time
import random
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
# Define all local file paths used by the pipeline.
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [ ]:
# ----------------------------
# File paths
# ----------------------------
from pathlib import Path

if "COLAB_GPU" in os.environ:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DATA_DIR = Path("/content/drive/MyDrive/climate-claim-verification/data")
else:
    # Local usage: notebook is inside notebooks/
    DATA_DIR = Path("../data")

TRAIN_FILE = DATA_DIR / "train-claims.json"
DEV_FILE = DATA_DIR / "dev-claims.json"
TEST_FILE = DATA_DIR / "test-claims-unlabelled.json"
EVIDENCE_FILE = DATA_DIR / "evidence.json"
EVAL_SCRIPT = DATA_DIR / "eval.py"

# EVIDENCE_GDRIVE_ID = "1JlUzRufknsHzKzvrEjgw8D3n_IRpjzo6"

# ----------------------------
# Reproducibility
# ----------------------------
RANDOM_SEED = 42
# The evidence file is large; this optional block downloads it only when it is missing.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ----------------------------
# Read a JSON file and return the parsed Python object.
# Download evidence.json if needed
# ----------------------------
# if not EVIDENCE_FILE.exists():
#     print("evidence.json not found. Downloading from Google Drive...")
#     !gdown --id 1JlUzRufknsHzKzvrEjgw8D3n_IRpjzo6 -O evidence.json
# else:
#     print("evidence.json already exists.")
# Load labelled train/dev claims, unlabelled test claims, and the full evidence corpus.

# ----------------------------
# Load data
# ----------------------------
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("\n--- Loading data ---")
train_claims  = load_json(TRAIN_FILE)
dev_claims    = load_json(DEV_FILE)
test_claims   = load_json(TEST_FILE)
evidence_dict = load_json(EVIDENCE_FILE)
# Retrieval hyperparameters selected from previous tuning experiments.

evidence_ids   = list(evidence_dict.keys())
evidence_texts = list(evidence_dict.values())

print(f"Train claims: {len(train_claims)}")
print(f"Dev claims:   {len(dev_claims)}")
print(f"Test claims:  {len(test_claims)}")
print(f"Evidence documents: {len(evidence_dict)}")

Mounted at /content/drive
Using device: cuda

--- Loading data ---
Train claims: 1228
Dev claims:   154
Test claims:  153
Evidence documents: 1208827


In [ ]:
# CrossEncoder training and inference settings for evidence reranking.
# ----------------------------
# Best retrieval config （from previous experiment ）
# ----------------------------
BEST_ALPHA = 0.03
BEST_GAP   = 0.03
BEST_MIN_K = 1
BEST_MAX_K = 6

TOP_K_A           = 250
TOP_K_B           = 250
FINAL_CANDIDATE_K = 400

# DeBERTa-based claim classifier settings.
NEGATIVE_PER_POSITIVE = 4
MAX_TRAIN_CLAIMS      = 3000

BASE_MODEL_NAME    = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CE_MAX_LENGTH      = 512
TRAIN_BATCH_SIZE   = 16
PREDICT_BATCH_SIZE = 64
EPOCHS             = 3
LEARNING_RATE      = 1e-5
WARMUP_RATIO       = 0.1

# ----------------------------
# Map string labels to integer IDs required by the classifier loss function.
# Classification settings
# ----------------------------
CLF_MODEL_NAME          = "cross-encoder/nli-deberta-v3-small"
CLF_MAX_LENGTH          = 512
CLF_TRAIN_EPOCHS        = 5
# Retriever A uses unigram TF-IDF with English stopword removal for precise lexical matching.
CLF_LEARNING_RATE       = 2e-5
CLF_BATCH_SIZE          = 8
CLF_WARMUP_RATIO        = 0.1
CLF_MAX_EVIDENCE_TOKENS = 300

LABEL2ID = {"SUPPORTS": 0, "REFUTES": 1, "NOT_ENOUGH_INFO": 2, "DISPUTED": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
print("\nLabel mapping:", LABEL2ID)


Label mapping: {'SUPPORTS': 0, 'REFUTES': 1, 'NOT_ENOUGH_INFO': 2, 'DISPUTED': 3}


In [ ]:
# ----------------------------
# TF-IDF Retriever A
# ----------------------------
print("\n--- Building TF-IDF retriever A ---")
# Retriever B uses unigram + bigram TF-IDF to capture short phrases and improve recall.
start_time = time.time()
vectorizer_a = TfidfVectorizer(
    token_pattern=r"(?u)\b[a-z0-9]+\b",
    lowercase=True,
    stop_words="english"
)
tfidf_matrix_a = vectorizer_a.fit_transform(evidence_texts)
print("TF-IDF A matrix shape:", tfidf_matrix_a.shape)
print(f"Build time: {time.time() - start_time:.2f}s")

# ----------------------------
# TF-IDF Retriever B
# ----------------------------
print("\n--- Building TF-IDF retriever B ---")
start_time = time.time()
vectorizer_b = TfidfVectorizer(
    token_pattern=r"(?u)\b[a-z0-9]+\b",
    lowercase=True,
# Return the indices of the highest-scoring items without fully sorting the whole evidence corpus.
    stop_words=None,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    max_features=500000
)
tfidf_matrix_b = vectorizer_b.fit_transform(evidence_texts)
# Scale scores to the 0-1 range so that scores from different retrievers can be combined.
print("TF-IDF B matrix shape:", tfidf_matrix_b.shape)
print(f"Build time: {time.time() - start_time:.2f}s")


--- Building TF-IDF retriever A ---
TF-IDF A matrix shape: (1208827, 531456)
Build time: 48.63s

--- Building TF-IDF retriever B ---
TF-IDF B matrix shape: (1208827, 500000)
Build time: 84.02s


In [ ]:
# ----------------------------
# Utility functions
# ----------------------------
def get_top_indices(scores, top_k):
    if top_k >= len(scores):
        return np.argsort(scores)[::-1]
# Run Retriever A for one claim and keep both raw and normalised scores for each candidate.
    top_indices = np.argpartition(scores, -top_k)[-top_k:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    return top_indices

def minmax_normalise(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return values
    v_min, v_max = float(np.min(values)), float(np.max(values))
    if abs(v_max - v_min) < 1e-12:
        return np.zeros_like(values)
    return (values - v_min) / (v_max - v_min)

def retrieve_a_with_scores(claim_text, top_k=TOP_K_A):
    claim_vec   = vectorizer_a.transform([claim_text])
    scores      = linear_kernel(claim_vec, tfidf_matrix_a).flatten()
    top_indices = get_top_indices(scores, top_k)
    raw_scores  = scores[top_indices]
    norm_scores = minmax_normalise(raw_scores)
# Run Retriever B for one claim using the broader unigram+bigram feature space.
    results = []
    for rank, (idx, raw_score, norm_score) in enumerate(
            zip(top_indices, raw_scores, norm_scores), start=1):
        results.append({
            "evidence_id":  evidence_ids[idx],
            "rank_a":       rank,
            "score_a":      float(raw_score),
            "norm_score_a": float(norm_score)
        })
    return results

def retrieve_b_with_scores(claim_text, top_k=TOP_K_B):
    claim_vec   = vectorizer_b.transform([claim_text])
    scores      = linear_kernel(claim_vec, tfidf_matrix_b).flatten()
    top_indices = get_top_indices(scores, top_k)
    raw_scores  = scores[top_indices]
    norm_scores = minmax_normalise(raw_scores)
    results = []
    for rank, (idx, raw_score, norm_score) in enumerate(
# Merge the two TF-IDF retrievers and attach ranking/score features used later for fusion.
            zip(top_indices, raw_scores, norm_scores), start=1):
        results.append({
            "evidence_id":  evidence_ids[idx],
            "rank_b":       rank,
            "score_b":      float(raw_score),
            "norm_score_b": float(norm_score)
        })
    return results

def retrieve_candidates_ensemble_with_features(
        claim_text, top_k_a=TOP_K_A, top_k_b=TOP_K_B, final_k=FINAL_CANDIDATE_K):
    results_a = retrieve_a_with_scores(claim_text, top_k_a)
    results_b = retrieve_b_with_scores(claim_text, top_k_b)
    candidate_info = {}
    for item in results_a:
        eid = item["evidence_id"]
        candidate_info[eid] = {
            "evidence_id":  eid, "rank_a": item["rank_a"], "rank_b": None,
            "score_a": item["score_a"], "score_b": 0.0,
            "norm_score_a": item["norm_score_a"], "norm_score_b": 0.0
        }
    for item in results_b:
# Reciprocal Rank Fusion rewards evidence that appears high in either retriever list.
        eid = item["evidence_id"]
        if eid not in candidate_info:
            candidate_info[eid] = {
                "evidence_id":  eid, "rank_a": None, "rank_b": item["rank_b"],
                "score_a": 0.0, "score_b": item["score_b"],
                "norm_score_a": 0.0, "norm_score_b": item["norm_score_b"]
            }
        else:
            candidate_info[eid]["rank_b"]       = item["rank_b"]
            candidate_info[eid]["score_b"]      = item["score_b"]
            candidate_info[eid]["norm_score_b"] = item["norm_score_b"]
# Interleave candidates from both retrievers to preserve diversity before cutting to final_k.
    for eid, info in candidate_info.items():
        rrf = 0.0
        if info["rank_a"] is not None:
            rrf += 1.0 / (60.0 + info["rank_a"])
        if info["rank_b"] is not None:
            rrf += 1.0 / (60.0 + info["rank_b"])
        info["rrf_score"]     = float(rrf)
        info["lexical_score"] = float(
            max(info["norm_score_a"], info["norm_score_b"]) + 5.0 * rrf)
    merged, seen = [], set()
    max_len = max(len(results_a), len(results_b))
    for i in range(max_len):
        if i < len(results_a):
            eid = results_a[i]["evidence_id"]
            if eid not in seen:
                seen.add(eid)
# Cache dev candidates once so later CrossEncoder scoring does not repeat TF-IDF retrieval.
                merged.append(candidate_info[eid])
        if i < len(results_b):
            eid = results_b[i]["evidence_id"]
            if eid not in seen:
                seen.add(eid)
                merged.append(candidate_info[eid])
        if len(merged) >= final_k:
            break
    return merged[:final_k]

In [ ]:
# ----------------------------
# Pre-compute dev AND train TF-IDF candidates
# Cache train candidates as well because classifier training uses retrieved evidence, not gold evidence.
# (train candidates needed later for classifier training)
# ----------------------------
print("\n--- Pre-computing dev TF-IDF candidates ---")
start_time = time.time()
dev_candidate_cache = {}
for claim_id, claim_info in tqdm(dev_claims.items(), desc="Dev TF-IDF retrieval"):
    claim_text = claim_info["claim_text"]
    dev_candidate_cache[claim_id] = {
        "claim_text":      claim_text,
        "candidate_infos": retrieve_candidates_ensemble_with_features(claim_text)
    }
print(f"Done in {time.time() - start_time:.2f}s")

print("\n--- Pre-computing train TF-IDF candidates ---")
start_time = time.time()
train_candidate_cache = {}
for claim_id, claim_info in tqdm(train_claims.items(), desc="Train TF-IDF retrieval"):
    claim_text = claim_info["claim_text"]
    train_candidate_cache[claim_id] = {
        "claim_text":      claim_text,
        "candidate_infos": retrieve_candidates_ensemble_with_features(claim_text)
    }
print(f"Done in {time.time() - start_time:.2f}s")


--- Pre-computing dev TF-IDF candidates ---


Dev TF-IDF retrieval:   0%|          | 0/154 [00:00<?, ?it/s]

Done in 242.19s

--- Pre-computing train TF-IDF candidates ---


Train TF-IDF retrieval:   0%|          | 0/1228 [00:00<?, ?it/s]

Done in 1859.53s


# 2. CrossEncoder Reranking + Evidence Selection

Fine-tune CrossEncoder, score candidates, fuse with lexical score, and select 1–6 evidence per claim via gap-based adaptive selection. Produces `dev_scored_cache` / `train_scored_cache`.

In [ ]:
# ============================================================
# 2. Model Implementation
# ============================================================

from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader

# ============================================================
# Part A: Retrieval CrossEncoder
# ============================================================

# Build binary training pairs for the CrossEncoder: gold evidence is positive, sampled evidence is negative.
print("\n--- Building CrossEncoder training pairs ---")
start_time = time.time()

# Each InputExample contains a claim-evidence pair and a relevance label.
train_samples    = []
train_items      = list(train_claims.items())
if MAX_TRAIN_CLAIMS is not None:
    train_items = train_items[:MAX_TRAIN_CLAIMS]
all_evidence_set = set(evidence_ids)

# Iterate through each training claim and create positive/negative evidence pairs.
for claim_id, claim_info in tqdm(train_items, desc="Pair construction"):
    claim_text        = claim_info["claim_text"]
    gold_evidence_ids = claim_info.get("evidences", [])
    gold_set          = set(gold_evidence_ids)
# Positive examples teach the model what truly relevant evidence looks like.
    for gold_id in gold_evidence_ids:
        if gold_id in evidence_dict:
            train_samples.append(
                InputExample(texts=[claim_text, evidence_dict[gold_id]], label=1.0))
# Negative examples are sampled from non-gold evidence to teach the model to reject irrelevant matches.
    n_negatives = len(gold_evidence_ids) * NEGATIVE_PER_POSITIVE
    if n_negatives <= 0:
        continue
    negative_pool = list(all_evidence_set - gold_set)
    for neg_id in random.sample(negative_pool, min(n_negatives, len(negative_pool))):
        train_samples.append(
            InputExample(texts=[claim_text, evidence_dict[neg_id]], label=0.0))

print(f"Training samples: {len(train_samples)}")
print(f"Time: {(time.time() - start_time) / 60:.2f} min")

# Fine-tune the CrossEncoder reranker on the constructed relevance pairs.
print("\n--- Fine-tuning CrossEncoder ---")
ce_model = CrossEncoder(BASE_MODEL_NAME, num_labels=1, max_length=CE_MAX_LENGTH, device=DEVICE)
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=TRAIN_BATCH_SIZE)
warmup_steps = int(len(train_dataloader) * EPOCHS * WARMUP_RATIO)

start_time = time.time()
ce_model.fit(
    train_dataloader=train_dataloader,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": LEARNING_RATE},
    show_progress_bar=True
)
print(f"CrossEncoder fine-tuning time: {(time.time() - start_time) / 60:.2f} min")

# ----------------------------
# Scoring / fusion / selection helpers
# ----------------------------
# Score all candidate evidence items for one claim using the fine-tuned CrossEncoder.
def score_claim_candidates_with_features(claim_text, candidate_infos):
    pairs = [[claim_text, evidence_dict.get(item["evidence_id"], "")]
             for item in candidate_infos]
    ce_scores = ce_model.predict(pairs, batch_size=PREDICT_BATCH_SIZE, show_progress_bar=False)
    scored_items = []
    for item, ce_score in zip(candidate_infos, ce_scores):
        new_item = dict(item)
        new_item["ce_score"] = float(ce_score)
        scored_items.append(new_item)
    scored_items.sort(key=lambda x: x["ce_score"], reverse=True)
    return scored_items

# Combine semantic CrossEncoder scores with a small lexical score contribution.
def rank_with_score_fusion(scored_items, alpha=BEST_ALPHA):
    if len(scored_items) == 0:
        return []
    ce_values  = np.array([x["ce_score"]     for x in scored_items], dtype=float)
    lex_values = np.array([x["lexical_score"] for x in scored_items], dtype=float)
# Normalise scores before fusion because CE and TF-IDF scores are on different scales.
    norm_ce    = minmax_normalise(ce_values)
    norm_lex   = minmax_normalise(lex_values)
    fused = []
    for item, cn, ln in zip(scored_items, norm_ce, norm_lex):
        new_item = dict(item)
        new_item["norm_ce_score"]      = float(cn)
        new_item["norm_lexical_score"] = float(ln)
# alpha controls how much lexical TF-IDF/RRF evidence contributes to the final ranking.
        new_item["final_score"]        = float((1.0 - alpha) * cn + alpha * ln)
        fused.append(new_item)
    fused.sort(key=lambda x: x["final_score"], reverse=True)
    return fused

# Select a variable number of evidences based on how close their scores are to the top item.
def select_evidences_by_gap(fused_items, gap=BEST_GAP, min_k=BEST_MIN_K, max_k=BEST_MAX_K):
    if len(fused_items) == 0:
        return []
    top_score = fused_items[0]["final_score"]
    selected  = [x["evidence_id"] for x in fused_items
                 if (top_score - x["final_score"]) <= gap]
    if len(selected) < min_k:
        selected = [x["evidence_id"] for x in fused_items[:min_k]]
    return selected[:max_k]

# ----------------------------
# Score dev candidates with CrossEncoder
# ----------------------------
# Apply the CrossEncoder to all dev candidates so retrieval quality can be evaluated.
print("\n--- Scoring dev candidates with CrossEncoder ---")
start_time = time.time()
dev_scored_cache = {}
for i, (claim_id, info) in enumerate(dev_candidate_cache.items(), start=1):
    dev_scored_cache[claim_id] = {
        "claim_text":   info["claim_text"],
        "scored_items": score_claim_candidates_with_features(
            info["claim_text"], info["candidate_infos"])
    }
    if i % 10 == 0 or i == len(dev_candidate_cache):
        elapsed = time.time() - start_time
        print(f"[Dev] {i}/{len(dev_candidate_cache)} | {elapsed/60:.1f}min | {elapsed/i:.2f}s/claim")
print(f"Dev scoring time: {(time.time() - start_time) / 60:.2f} min")

# ----------------------------
# Score train candidates with CrossEncoder
# classifier trains on the SAME evidence distribution as inference
# ----------------------------
# Score train candidates too; the classifier should train on the same retrieved-evidence distribution used at inference.
print("\n--- Scoring train candidates with CrossEncoder ---")
start_time = time.time()
train_scored_cache = {}
for i, (claim_id, info) in enumerate(train_candidate_cache.items(), start=1):
    train_scored_cache[claim_id] = {
        "claim_text":   info["claim_text"],
        "scored_items": score_claim_candidates_with_features(
            info["claim_text"], info["candidate_infos"])
    }
    if i % 50 == 0 or i == len(train_candidate_cache):
        elapsed = time.time() - start_time
        print(f"[Train] {i}/{len(train_candidate_cache)} | {elapsed/60:.1f}min | {elapsed/i:.2f}s/claim")
print(f"Train scoring time: {(time.time() - start_time) / 60:.2f} min")

# 3. Export Retrieved Evidence

Dump per-claim retrieved evidence + label for train/dev, to be consumed by the LLM classifier notebook.

In [ ]:
# ============================================================
# Export retrieved evidence for the LLM classifier
# 导出检索结果:每条 claim 的 (文本 + 选出的证据 + 标签)
# ============================================================
import json, shutil

def export_retrieved_for_clf(scored_cache, claims_dict, out_path):
    data = {}
    for claim_id, info in scored_cache.items():
        fused    = rank_with_score_fusion(info["scored_items"])
        ret_eids = select_evidences_by_gap(fused)
        data[claim_id] = {
            "claim_text":     info["claim_text"],
            "evidence_ids":   ret_eids,
            "evidence_texts": [evidence_dict.get(e, "") for e in ret_eids],
            "label":          claims_dict[claim_id].get("claim_label"),
        }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(data)} -> {out_path}")
    return data

train_export = export_retrieved_for_clf(train_scored_cache, train_claims, "retrieved_train.json")
dev_export   = export_retrieved_for_clf(dev_scored_cache,   dev_claims,   "retrieved_dev.json")

# Persist to Google Drive so the LLM classifier notebook can load them directly
if "COLAB_GPU" in os.environ:
    for fn in ["retrieved_train.json", "retrieved_dev.json"]:
        shutil.copy(fn, "/content/drive/MyDrive/climate-claim-verification/" + fn)
    print("Copied to Drive.")

# sanity check
sample_id = next(iter(dev_export))
print("\nSample:", sample_id)
print(dev_export[sample_id])
